In [0]:
%run ../lib/ingestion_functions

In [0]:
container_source = 'silver'
source_table = 'cidades'
source_schema = 'bronze'
directory_source = 'novadrive'
subdirectory_source ='incrementalLoad'
file_source = 'cidade'
container_target = 'silver'
directory_target = 'novadrive'
delta_table_name = 'silver_cidade'
id_field = 'id_cidades'
timestamp_field = 'N/A'

In [0]:
#source_path = f"abfss://raw@adlsnovadriveeusdev.dfs.core.windows.net/{directory_source}/{subdirectory_source}/{file_source}/"
target_path = f"abfss://silver@adlsnovadriveeusdev.dfs.core.windows.net/novadrive/{delta_table_name}"
schema_location = f"abfss://metastore@adlsnovadriveeusuc.dfs.core.windows.net/uc-metastore-novadrive-eus/dev/silver/{delta_table_name}_schema"
checkpoint_location = f"abfss://metastore@adlsnovadriveeusuc.dfs.core.windows.net/uc-metastore-novadrive-eus/dev/silver/{delta_table_name}_chk"

In [0]:
bronzeSelect = f"""
            select * from dev.bronze.cidades
        """
bronzeInfo = spark.sql(bronzeSelect)

In [0]:
creator = DeltaTableCreator(spark)

creator.create_table_with_cdf(
    df=bronzeInfo,
    catalog="dev",
    schema="silver",
    table=delta_table_name,
    path=target_path,
    mode="overwrite"  # sobrescreve dados no path se já existirem
)


In [0]:
 spark, data_format, target_path, catalog, schemaname, tablename, schema_location,checkpoint_location, id_field, timestamp_field

In [0]:
table_exists = spark.catalog.tableExists(f"dev.silver.{delta_table_name}")
if not table_exists:

    dbutils.fs.rm(checkpoint_location, True)
    dbutils.fs.rm(schema_location, True)

    ingestor = IngestionCDF(
    spark=spark,
    data_format="delta",
    target_path=target_path,
    schema_location=schema_location,
    checkpoint_location=checkpoint_location,
    source_table = source_table,
    source_schema = source_schema,
    catalog="dev",
    schemaname="silver",
    id_field=id_field,
    timestamp_field=timestamp_field,
    tablename=delta_table_name
)

    
    print(f"Tabela {delta_table_name} não existe ou a carga é Full Load, criando...")

    ingestor.fullLoadSilver(delta_table_name)
   
else:

    print(f"Table {delta_table_name} already exists, doing the upsert...")

    # incrementalIngestor = IncrementalIngestor(
    #     spark=spark,
    #     source_path=source_path,
    #     data_format="parquet",
    #     target_path=target_path,
    #     schema_location=schema_location,
    #     checkpoint_location=checkpoint_location,
    #     catalog="dev",
    #     schemaname="bronze",
    #     id_field=id_field,
    #     timestamp_field=timestamp_field,
    #     tablename=delta_table_name
    #  )
    # incrementalIngestor.executeLoadAndSave(source_path)

In [0]:
%sql

select * from dev.bronze.cidades

In [0]:
%sql

delete from dev.silver.silver_cidade
where id_estados=1

In [0]:
%sql

select * from dev.silver.silver_cidade

In [0]:
%sql

DESCRIBE HISTORY dev.silver.silver_cidade

In [0]:
display(spark.sql(
    
))

In [0]:
%sql

describe detail dev.silver.silver_cidade